<a href="https://colab.research.google.com/github/Monocyte572/CNN-code-to-use-/blob/main/Module_1B_AI_Guided_Decision_Making_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1B — AI-Guided Decision-Making
## Self-Driving Labs Workshop | Coding Notebook

**Activity time:** ~25 minutes
**Coding level:** No advanced coding required — most of the code is already written for you.

---

### ⚠️ Important note on the dataset

This notebook uses a **small, synthetic medicinal-chemistry dataset**.

> **This is a teaching dataset, not a drug-discovery dataset.**

We're using synthetic data on purpose as it lets us see the entire decision-making pipeline clearly, without getting lost in the complexity of a real chemistry problem. Nothing you see here should be interpreted as real chemical or biological insight.

### What you'll do

You will move through six steps:

1. **Scientific information** — a set of candidate molecules and what we know about them
2. **Machine-readable representation** — turning molecules into numbers
3. **AI prediction** — a simple model predicts an outcome
4. **Prediction uncertainty** — how confident is the model, really?
5. **Scientific decision** — combine prediction, uncertainty, and practical constraints
6. **Next experiment** — you decide what to run next, and explain why

> **The important part isn't the model. The important part is what happens *after* the model gives you a prediction.**




## Setup

Run the cell below to load the libraries we'll use. Nothing to edit here.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C

RNG = np.random.default_rng(42)
pd.set_option("display.precision", 2)

print("Libraries loaded. You're ready to go.")


Libraries loaded. You're ready to go.


## Step 1 — Scientific Information

**The scenario:** You are screening a small library of hypothetical molecules for their ability to inhibit a target enzyme. For each molecule that has *already been tested in the lab*, we know:

- Molecular weight (MW)
- LogP (a measure of "greasiness" / lipophilicity)
- Number of hydrogen-bond donors (HBD)
- Number of hydrogen-bond acceptors (HBA)
- Number of rotatable bonds (RotB)
- The measured outcome: **pIC50** (higher = more potent inhibitor)

This is the "scientific information" — the raw facts a chemist would have in a lab notebook, before any modeling happens.

Run the cell below to generate and view the training data (synthetic, but structured like a real assay dataset).

In [ ]:
def make_training_set(n=40):
    mw   = RNG.normal(350, 60, n).clip(150, 550)
    logp = RNG.normal(2.5, 1.2, n).clip(-1, 6)
    hbd  = RNG.integers(0, 5, n)
    hba  = RNG.integers(1, 9, n)
    rotb = RNG.integers(0, 10, n)

    # Synthetic "ground truth" relationship (unknown to the model, known to us as workshop designers)
    pIC50 = (
        6.0
        + 0.9 * np.exp(-((logp - 2.8) ** 2) / 2.0)   # sweet spot around logp ~ 2.8
        - 0.004 * np.abs(mw - 350)
        - 0.15 * hbd
        - 0.05 * rotb
        + RNG.normal(0, 0.35, n)                      # experimental noise
    )

    return pd.DataFrame({
        "MW": mw.round(1),
        "LogP": logp.round(2),
        "HBD": hbd,
        "HBA": hba,
        "RotB": rotb,
        "pIC50": pIC50.round(2),
    })

train_df = make_training_set()
train_df.index = [f"M{idx+1:02d}" for idx in range(len(train_df))]
train_df.index.name = "Molecule"

print(f"{len(train_df)} previously-tested molecules:")
train_df.head(10)


## Step 2 — Machine-Readable Representation

**What does the AI actually see?**

```
MOLECULE  →  MACHINE-READABLE REPRESENTATION  →  FEATURES  →  MODEL  →  PREDICTION
```

A chemist looks at a molecule and sees structure, chemistry, and context. A model doesn't — it needs numbers. Here, we're using simple **molecular descriptors** (MW, LogP, HBD, HBA, RotB) as a stand-in for a real machine-readable representation.

This is deliberately simplified. Tomorrow, in Module 2A and 2B, you'll go much deeper into how molecular representations are built, how features are generated, and how models are trained and evaluated. For today, just notice the shape of the pipeline.

Run the cell below to separate the **features** (what the model sees) from the **target** (what we're trying to predict).

In [ ]:
FEATURE_COLUMNS = ["MW", "LogP", "HBD", "HBA", "RotB"]
TARGET_COLUMN = "pIC50"

X_train = train_df[FEATURE_COLUMNS].values
y_train = train_df[TARGET_COLUMN].values

print("Features (what the model sees):")
display(train_df[FEATURE_COLUMNS].head())

print("\nTarget (what we're trying to predict):")
display(train_df[[TARGET_COLUMN]].head())


## Step 3 — AI Prediction

Now we hand the features and targets to a simple model. We're using a **Gaussian Process Regressor** — a model chosen specifically because it gives us a prediction *and* a principled estimate of uncertainty for each new molecule, which we'll need in Step 4.

You don't need to understand the internals of the model. Just notice: it learns a pattern from the 40 previously-tested molecules, and now we're going to ask it to predict the outcome for molecules that **haven't been tested yet**.

Run the cell below to train the model and generate a candidate set of untested molecules.

In [ ]:
# Standardize features (helps the model treat each descriptor fairly)
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)

kernel = C(1.0, (1e-2, 1e2)) * RBF(length_scale=np.ones(X_train.shape[1]), length_scale_bounds=(1e-1, 1e2)) \
         + WhiteKernel(noise_level=0.2, noise_level_bounds=(1e-3, 1.0))

model = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=3, random_state=0)
model.fit(X_train_scaled, y_train)

# --- A set of CANDIDATE molecules that have NOT been tested yet ---
candidates_df = pd.DataFrame({
    "MW":   [310, 480, 340, 355, 520, 300, 365, 410],
    "LogP": [2.7, 1.0, 4.8, 2.9, 0.2, 3.6, 2.6, 5.5],
    "HBD":  [1,   4,   0,   1,   3,   2,   1,   0],
    "HBA":  [4,   7,   2,   5,   8,   3,   5,   1],
    "RotB": [3,   6,   8,   2,   5,   4,   3,   9],
}, index=[f"C{i+1}" for i in range(8)])
candidates_df.index.name = "Candidate"

X_cand_scaled = scaler.transform(candidates_df[FEATURE_COLUMNS].values)
pred_mean, pred_std = model.predict(X_cand_scaled, return_std=True)

candidates_df["Predicted_pIC50"] = pred_mean.round(2)
print("Model trained. Predictions for 8 untested candidate molecules:")
candidates_df[FEATURE_COLUMNS + ["Predicted_pIC50"]]


## Step 4 — Prediction Uncertainty

A prediction on its own can be misleading. Two candidates might have the *same* predicted value, but very different confidence behind that number — usually because one is similar to molecules the model has already seen, and the other is not.

Run the cell below to attach an uncertainty estimate to each prediction and visualize it.

In [ ]:
candidates_df["Uncertainty"] = pred_std.round(2)

fig, ax = plt.subplots(figsize=(8, 4.5))
order = candidates_df["Predicted_pIC50"].sort_values(ascending=False).index
ax.errorbar(
    candidates_df.loc[order, "Predicted_pIC50"],
    range(len(order)),
    xerr=candidates_df.loc[order, "Uncertainty"],
    fmt="o", color="#2b6cb0", ecolor="#a0aec0", elinewidth=3, capsize=4, markersize=8,
)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order)
ax.set_xlabel("Predicted pIC50 (higher = more potent)")
ax.set_title("Predicted outcome ± model uncertainty, per candidate")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

candidates_df[["Predicted_pIC50", "Uncertainty"]].sort_values("Predicted_pIC50", ascending=False)


**Take a moment before moving on:**

- Which candidate has the *highest* predicted value?
- Is that also the candidate you're most *confident* about?
- What might explain a candidate having high uncertainty (large error bar)?

## Step 5 — Prediction ≠ Decision

A prediction is not a decision. Real scientific decisions also depend on **practical constraints** — things the model doesn't know about at all.

Run the cell below to add two more pieces of information to the table:

- **Experimental cost** (relative units — higher means more expensive/time-consuming to run)
- **Material availability** (whether you currently have enough compound in hand to run the experiment)

In [ ]:
candidates_df["Experimental_Cost"] = [3, 5, 2, 3, 6, 2, 3, 4]
candidates_df["Material_Available"] = ["Yes", "Yes", "No", "Yes", "No", "Yes", "Yes", "Limited"]

decision_table = candidates_df[
    ["Predicted_pIC50", "Uncertainty", "Experimental_Cost", "Material_Available"]
].sort_values("Predicted_pIC50", ascending=False)

print("Full decision table — this is everything you have to work with:")
decision_table


### Now make a scientific decision

**Which experiment would you run next?**

Don't just pick the candidate with the highest predicted value. Consider:

- **What is your objective?** (e.g., find the single most potent molecule vs. reduce uncertainty in a promising region vs. get a quick, cheap result)
- **What information influenced your choice?** (prediction, uncertainty, cost, availability — or some combination)
- **How did uncertainty affect your decision?** Did you favor a confident-but-moderate candidate, or a high-but-uncertain one?
- **What role should the scientist retain?** What is the AI good at here, and what does it need a human to weigh in on?

There isn't necessarily one universally correct answer. What matters is whether you can explain *why* the AI's output does — or doesn't — lead you toward a particular experimental action.

## Step 6 — Your Decision

Fill in three fields: chosen_candidate, objective, rationale (one sentence)
Run the cell
The notebook will immediately tell you:
How your pick compares to the highest-predicted candidate
Whether its uncertainty is above or below the group median
Whether material is fully available, limited, or unavailable
Whether it's also the cheapest option on the table

Don't just chase the highest number. Use the feedback to check your own reasoning
If the feedback surfaces a tension (e.g., "a higher-predicted candidate exists"), that's not an error as you will decide whether your reason for passing on it still holds






In [ ]:
# ---- YOUR ANSWER ----

chosen_candidate = ""   # e.g. "C1" — which candidate would you test next?

objective = ""           # What are you optimizing for with this choice?

rationale = ""           # One sentence: why this candidate, and not another?

# ---- FEEDBACK (just run this — nothing to edit below) ----

if chosen_candidate not in decision_table.index:
    print("Enter one of the candidate IDs above (e.g. \"C1\") to see feedback on your choice.")
else:
    row = decision_table.loc[chosen_candidate]
    best_pred = decision_table["Predicted_pIC50"].idxmax()
    best_uncertainty = decision_table["Uncertainty"].idxmin()
    cheapest = decision_table["Experimental_Cost"].idxmin()

    print(f"You chose {chosen_candidate}:")
    print(f"  Predicted pIC50: {row['Predicted_pIC50']}  |  Uncertainty: {row['Uncertainty']}  |  "
          f"Cost: {row['Experimental_Cost']}  |  Material available: {row['Material_Available']}")
    print()

    if chosen_candidate == best_pred:
        print("- This is the candidate with the highest predicted value on the table.")
    else:
        gap = decision_table.loc[best_pred, "Predicted_pIC50"] - row["Predicted_pIC50"]
        print(f"- {best_pred} has a higher predicted value (by {gap:.2f}). Was there a reason you passed on it?")

    if chosen_candidate == best_uncertainty:
        print("- This is also the candidate the model is most confident about.")
    elif row["Uncertainty"] > decision_table["Uncertainty"].median():
        print("- This candidate carries above-median uncertainty — the model is less sure here than for most others.")
    else:
        print("- This candidate's uncertainty is on the lower side, relative to the full set.")

    if row["Material_Available"] != "Yes":
        print(f"- Material availability is \"{row['Material_Available']}\" — this could delay or block the experiment.")

    if chosen_candidate == cheapest:
        print("- This is also the cheapest experiment on the table.")

    print(f"\nYour objective: {objective or '[not yet filled in]'}")
    print(f"Your rationale: {rationale or '[not yet filled in]'}")


---

## Wrap-up

You've now walked through the full pipeline:

**Scientific information → machine-readable representation → AI prediction → prediction uncertainty → scientific decision → next experiment**



